# data — Direction 1: the replica caches the training reads

One replica cache per scenario, built on a CPU (`hvac_controller/d1/cache.py`): spec, ROM, exogenous series, policy
metadata and action spec, pickled once so that training and evaluation never rebuild the ROM. Each cache's JSON
logs the Jacobi bound q of the fast air solve (must stay < 0.5).

Modes. `N_SHARDS` sessions run in parallel with `RUN_ID = 0 … N_SHARDS - 1`; each takes every N_SHARDS-th scenario.
- `all`: the 30 shipped scenarios.
- `gen`: the generated scenarios (`hvac_controller/d1/augment.py`), 86 in all: 24 generated buildings × 3 TMYx
  strips (training), 6 held-out buildings × 2 strips of held-out stations, building_train_2 × minneapolis_1/_5.
  The buildings come from the ranges the shipped six lie in (V2's came from the generator's defaults, 2–3 × their
  footprint). Each session builds the same manifest and buildings, then its share of the weather presets,
  scenarios and caches.

**Inputs:** the competition data; `weathertmyx` for `gen`. CPU, Internet on.

**Output:** `/kaggle/working/buildings_<mode>_<run>.zip`, flat: the caches (`*.pkl` + `*.json`); for `gen` also the
manifest, the generated buildings' YAMLs and the held-out scenarios' weather presets. Merged, the sessions' zips make
the `buildings` dataset (docs/KAGGLE.md).

In [ ]:
REPO_URL = "https://github.com/CryAndRRich/hvac-controller"
REPO_REF = "direction1"
MODE = "gen"              # "all" or "gen"
RUN_ID, N_SHARDS = 0, 2
N_JOBS = 2                # cache builds at once on the 4 CPUs (3 at once ran out of memory from ~570k cells)

In [ ]:
import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

REPO, INPUT, WORK = Path("/tmp/hvac-controller"), Path("/kaggle/input"), Path("/tmp/work")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--quiet", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", REPO_REF], check=True)
print(subprocess.run(["git", "-C", str(REPO), "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
sys.path.insert(0, str(REPO))
ENV = dict(os.environ, PYTHONPATH=str(REPO))


def dataset(slug):
    """One of the four datasets (docs/KAGGLE.md), wherever Kaggle mounts it."""
    return next(p for p in (INPUT / "datasets" / "cryandrrich" / slug, INPUT / slug) if p.is_dir())


SHIPPED = next(p for p in sorted(INPUT.rglob("training_scenarios"))
               if (p / "building_train_1" / "atlanta_2" / "config.gin").exists())
ROOT = REPO / "smart_buildings" / "training_scenarios"
shutil.copytree(SHIPPED, ROOT, dirs_exist_ok=True)
CACHES, LOGS = WORK / "caches", WORK / "logs"  # outside /kaggle/working: the output is the zip alone
for d in (CACHES, LOGS):
    d.mkdir(parents=True, exist_ok=True)
EXTRA = []  # the zip's files besides the caches

if MODE == "all":
    SCENARIOS = sorted(f"{p.parent.parent.name}/{p.parent.name}" for p in ROOT.glob("*/*/config.gin"))[RUN_ID::N_SHARDS]
else:
    from hvac_controller import weather

    PACKED = weather.find_packed(dataset("weathertmyx"))
    MANIFEST = WORK / "manifest.json"
    steps = [["manifest", str(MANIFEST), "--packed", str(PACKED)], ["buildings", str(MANIFEST)]]
    SHARD = WORK / f"shard_{RUN_ID}.json"
    t0 = time.time()
    for cmd in steps + [["scenarios", str(MANIFEST), "--packed", str(PACKED), "--shard", str(RUN_ID),
                         "--shards", str(N_SHARDS), "--out", str(SHARD)]]:
        r = subprocess.run([sys.executable, "-m", "hvac_controller.d1.augment", *cmd], cwd=REPO, env=ENV,
                           capture_output=True, text=True)
        print(f"[{time.time() - t0:5.0f}s] augment {cmd[0]}: exit {r.returncode}\n" + (r.stdout + r.stderr)[-3000:])
        assert r.returncode == 0, cmd
    ENTRIES = json.loads(SHARD.read_text())
    SCENARIOS = [e["label"].replace("__", "/", 1) for e in ENTRIES]
    # evaluate.ipynb rebuilds the held-out scenarios for the organisers' simulator from these
    EXTRA = [MANIFEST] + sorted((REPO / "smart_buildings" / "buildings").glob("building_[0-9]*.yaml"))
    for e in ENTRIES:
        if e["tag"] == "gen_val":
            EXTRA += [REPO / "smart_buildings" / "weather" / f"{e['weather']}.yaml",
                      REPO / "smart_buildings" / "weather" / "csv" / f"{e['weather']}.csv"]
missing = [s for s in SCENARIOS if not (ROOT / s / "config.gin").exists()]
assert not missing, missing
print(len(SCENARIOS), "scenarios here:", SCENARIOS)

In [ ]:
# Caches, N_JOBS at a time; each build's log is printed when it ends.
threads = str(max(1, (os.cpu_count() or 4) // N_JOBS))
env = dict(ENV, OMP_NUM_THREADS=threads, OPENBLAS_NUM_THREADS=threads, MKL_NUM_THREADS=threads)
t0 = time.time()
pending, running = list(SCENARIOS), []
while pending or running:
    while pending and len(running) < N_JOBS:
        tag = pending.pop(0).replace("/", "__")
        log = open(LOGS / f"{tag}.txt", "w")
        cmd = [sys.executable, "-m", "hvac_controller.d1.cache", str(CACHES / f"{tag}.pkl"),
               "--config", str(ROOT / tag.replace("__", "/", 1) / "config.gin")]
        running.append((tag, subprocess.Popen(cmd, cwd=REPO, env=env, stdout=log, stderr=subprocess.STDOUT), log))
    for item in list(running):
        tag, proc, log = item
        if proc.poll() is not None:
            log.close()
            running.remove(item)
            print(f"[{time.time() - t0:5.0f}s] {tag} exit {proc.returncode}: {(LOGS / f'{tag}.txt').read_text().strip()[-1500:]}",
                  flush=True)
    time.sleep(5)

In [ ]:
# Gate and the zip.
infos = {p.stem: json.loads(p.read_text()) for p in sorted(CACHES.glob("*.json"))}
for tag, i in infos.items():
    print(f"{tag:36s} Z={i['n_zones']:3d} V={i['n_vavs']:3d} A={i['n_ahus']:2d} obs={i['n_obs']} r={i['order']} "
          f"h={i['h']:.2f} q={i['jacobi_q']:.3f} sweeps={i['sweeps_float32']} | ROM {i['rom_s']:.0f}s exo {i['exo_s']:.0f}s")
print(f"\nGATE: {len(infos)}/{len(SCENARIOS)} caches, all q < 0.5:",
      len(infos) == len(SCENARIOS) and all(i["jacobi_q"] < 0.5 for i in infos.values()))
files = sorted(CACHES.glob("*.pkl")) + sorted(CACHES.glob("*.json")) + EXTRA
assert len({p.name for p in files}) == len(files), "flat zip: duplicate file names"
name = Path(f"/kaggle/working/buildings_{MODE}_{RUN_ID}.zip")
with zipfile.ZipFile(name, "w", zipfile.ZIP_DEFLATED) as z:
    for p in files:
        z.write(p, p.name)
print("wrote", name, f"{name.stat().st_size / 2 ** 20:.0f} MB")